# الذكاء الاصطناعي في الهندسة المدنية — الفصل 17 LLMs & RAG
Companion notebook. Run all cells (Runtime → Run all). Every number in the book comes from this code.

In [ ]:
!pip -q install xgboost shap scikit-image statsmodels ucimlrepo
import os; os.makedirs('fig', exist_ok=True)

In [ ]:
%%writefile ch17_corpus.py
# Original educational Arabic snippets written for this book (illustrative values, NOT quotations from any code).
DOCS = [
 "الغطاء الخرساني للحديد في الأعمدة الداخلية لا يقل عن 40 مم.",
 "الغطاء الخرساني للحديد في الأساسات الملامسة للتربة لا يقل عن 75 مم.",
 "الغطاء الخرساني للبلاطات الداخلية لا يقل عن 20 مم.",
 "في البيئات البحرية يزاد الغطاء الخرساني إلى 65 مم على الأقل للعناصر المعرضة لرذاذ البحر.",
 "تبدأ معالجة الخرسانة بالماء فور تصلب السطح وتستمر سبعة أيام على الأقل للإسمنت البورتلاندي العادي.",
 "في الطقس الحار تغطى الخرسانة بالخيش المبلل لتقليل التبخر خلال الأيام الأولى من المعالجة.",
 "لا يجوز صب الخرسانة إذا زادت درجة حرارة الخلطة عند الصب عن 32 درجة مئوية إلا بإجراءات خاصة.",
 "في الطقس البارد لا تصب الخرسانة إذا كانت درجة حرارة الجو أقل من 5 درجات مئوية دون حماية من الصقيع.",
 "يقاس هبوط الخرسانة الطازجة بمخروط أبرامز، ويؤخذ الهبوط المطلوب من مواصفات المشروع.",
 "يرفض الخلط إذا اختلف الهبوط المقاس عن الهبوط المحدد بأكثر من 25 مم.",
 "تؤخذ عينات المكعبات بمعدل مجموعة واحدة لكل 50 متراً مكعباً أو لكل يوم صب أيهما أكثر.",
 "تفحص المكعبات القياسية بعمر 7 أيام و28 يوماً، ويعتمد القبول على نتيجة 28 يوماً.",
 "تعتبر الخرسانة مقبولة إذا لم يقل متوسط كل ثلاث نتائج متتالية عن المقاومة المميزة.",
 "تحفظ المكعبات بعد فكها في أحواض ماء بدرجة حرارة 20 درجة مئوية تقريباً حتى موعد الفحص.",
 "عند الشك في مقاومة الخرسانة المصبوبة تؤخذ لباب من العنصر وتفحص بعد معايرتها.",
 "يستخدم مطرقة الارتداد للفحص السريع غير الإتلافي ولا يعتمد عليها وحدها لتقدير المقاومة.",
 "يستخدم جهاز الموجات فوق الصوتية لقياس سرعة النبضة وتقدير تجانس الخرسانة.",
 "لا تفك قوالب جوانب الجسور قبل 24 ساعة من الصب.",
 "لا تفك الدعامات السفلية للبلاطات قبل 14 يوماً إلا إذا أثبتت الفحوص بلوغ المقاومة المطلوبة.",
 "يجب ارتداء الخوذة والحذاء الواقي في جميع مناطق العمل داخل الموقع.",
 "يستخدم حزام الأمان الكامل عند العمل على ارتفاع يزيد على مترين.",
 "تسيج الحفريات التي يزيد عمقها على 1.5 متر وتوضع لافتات تحذيرية حولها.",
 "تدعم جوانب الحفريات أو تميل بزاوية آمنة إذا زاد العمق عن 1.5 متر في التربة غير المتماسكة.",
 "تخزن أكياس الإسمنت على منصات مرتفعة عن الأرض ومغطاة لحمايتها من الرطوبة.",
 "لا يستخدم الإسمنت الذي تكتل أو مضى على تصنيعه مدة طويلة دون فحصه.",
 "يجب أن تكون مياه الخلط صالحة للشرب أو مطابقة لمتطلبات الفحص الكيميائي.",
 "يمنع إضافة الماء إلى الخلطة في الموقع لزيادة قابلية التشغيل دون موافقة المهندس.",
 "يستخدم الهزاز الميكانيكي لدمك الخرسانة ويسحب ببطء لتجنب تكون الفراغات.",
 "لا يزيد ارتفاع السقوط الحر للخرسانة عند الصب على متر ونصف لتجنب انفصال الحصويات.",
 "تنظف أسطح الفواصل الإنشائية وتخشن قبل استئناف الصب.",
 "يتم وصل حديد التسليح بالتراكب بطول لا يقل عن الطول المحسوب في المخططات.",
 "يجب أن يكون حديد التسليح نظيفاً من الصدأ المتقشر والزيوت قبل الصب.",
 "توضع كراسي ومباعدات لضمان الغطاء الخرساني المطلوب أثناء الصب.",
 "يفحص حديد التسليح بأخذ عينات لاختبار الشد والانحناء من كل شحنة.",
 "يجب ألا يتجاوز عرض الشقوق في العناصر المعرضة للمياه 0.2 مم في حالة الخدمة.",
 "يسجل في يومية الموقع تاريخ الصب ودرجة الحرارة وعدد العينات المأخوذة.",
]
# 30 questions: (question, index of the snippet that answers it); several use synonyms, dialect, or different word forms
QA = [
 ("كم سماكة التغطية فوق التسليح في عمود داخل المبنى؟", 0),
 ("قديش الغطاء المطلوب للقواعد اللي على التراب؟", 1),
 ("ما الحد الأدنى لتغطية حديد السقف الداخلي؟", 2),
 ("مبنى على شاطئ البحر، كم يجب أن يكون الكفر؟", 3),
 ("كم يوم لازم نرش الخرسانة بالمي بعد الصب؟", 4),
 ("كيف نحمي الصبة من الشمس في الصيف؟", 5),
 ("ما أعلى حرارة مسموحة للباطون عند الصب؟", 6),
 ("هل يجوز الصب في الشتاء والحرارة تحت الصفر؟", 7),
 ("كيف نقيس قوام الخرسانة الطرية؟", 8),
 ("متى نرفض شاحنة الخرسانة بسبب السلامب؟", 9),
 ("كم عينة مكعبات نأخذ لكل كمية صب؟", 10),
 ("في أي عمر تكسر المكعبات؟", 11),
 ("ما شرط قبول نتائج الضغط؟", 12),
 ("أين نحفظ العينات بعد إخراجها من القالب؟", 13),
 ("نتائج المكعبات ضعيفة، كيف نتأكد من الخرسانة في العنصر نفسه؟", 14),
 ("هل يكفي الشميت هامر لمعرفة المقاومة؟", 15),
 ("ما فائدة فحص سرعة الموجات فوق الصوتية؟", 16),
 ("متى نشيل الطوبار عن جوانب الجسر؟", 17),
 ("متى يمكن إزالة القوائم تحت السقف؟", 18),
 ("شو معدات السلامة الشخصية الإلزامية في الموقع؟", 19),
 ("متى يلزم العامل بالحزام عند العمل على السقالة؟", 20),
 ("هل نضع سياجاً حول الحفرة العميقة؟", 21),
 ("كيف نمنع انهيار جدار الحفرية؟", 22),
 ("كيف نخزن الشوالات في المستودع؟", 23),
 ("إسمنت قديم ومتحجر، هل نستعمله؟", 24),
 ("هل يصلح ماء البئر لعمل الخلطة؟", 25),
 ("السائق بدو يزيد مي على الخلطة، مسموح؟", 26),
 ("كيف نستعمل الفايبريتر بشكل صحيح؟", 27),
 ("من أي ارتفاع مسموح نكب الخرسانة؟", 28),
 ("ما الحد الأقصى لعرض التشقق في خزان مياه؟", 34),
]


In [ ]:
import re, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from ch17_corpus import DOCS, QA

def norm_ar(s):
    s = re.sub(r"[ً-ْـ]", "", s)            # diacritics, tatweel
    s = re.sub("[إأآا]", "ا", s); s = s.replace("ى", "ي").replace("ة", "ه")
    return s
def tok(s): return re.findall(r"\w+", s)

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.D = [tok(d) for d in docs]; self.k1, self.b = k1, b
        self.avg = np.mean([len(d) for d in self.D]); N = len(self.D)
        df = {}
        for d in self.D:
            for w in set(d): df[w] = df.get(w, 0) + 1
        self.idf = {w: np.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
    def scores(self, q):
        s = np.zeros(len(self.D))
        for i, d in enumerate(self.D):
            L = len(d)
            for w in tok(q):
                f = d.count(w)
                if f: s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * L / self.avg))
        return s

def tfidf_ret(analyzer, ngram, pre=None):
    v = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, preprocessor=pre, sublinear_tf=True)
    X = v.fit_transform(DOCS)
    return lambda q: (X @ v.transform([q]).T).toarray().ravel()

bm = BM25(DOCS)
bmn = BM25([norm_ar(d) for d in DOCS])
retrievers = {
    "Word TF-IDF": tfidf_ret("word", (1, 1)),
    "BM25 (words)": bm.scores,
    "BM25 + Arabic normalisation": lambda q: bmn.scores(norm_ar(q)),
    "Char 2-4 TF-IDF": tfidf_ret("char_wb", (2, 4)),
    "Char 2-4 + normalisation": tfidf_ret("char_wb", (2, 4), pre=norm_ar),
}
res = {}
print("Chapter 17 RAG retrieval: %d original snippets, %d questions" % (len(DOCS), len(QA)))
for name, f in retrievers.items():
    h1 = h3 = 0; miss = []
    for q, gold in QA:
        s = f(q); order = np.argsort(-s)
        h1 += order[0] == gold; h3 += gold in order[:3]
        if gold not in order[:3]: miss.append(q)
    res[name] = (h1 / len(QA), h3 / len(QA), miss)
    print("  %-28s hit@1=%.2f  hit@3=%.2f  misses(top3)=%d" % (name, h1 / len(QA), h3 / len(QA), len(miss)))
best = "Char 2-4 + normalisation"
print("  questions missed by best retriever:")
for q in res[best][2]: print("   -", q)

# numeric guard: check every number+unit in an answer against the retrieved snippet
def numbers(s):
    return set(re.findall(r"(\d+(?:\.\d+)?)\s*(مم|متر|درجة|يوم|أيام|ساعة|متراً)", s))
cases = [
    ("كم الغطاء الخرساني للأساسات الملامسة للتربة؟", "الحد الأدنى للغطاء في الأساسات الملامسة للتربة هو 75 مم."),
    ("كم الغطاء الخرساني للأساسات الملامسة للتربة؟", "الحد الأدنى للغطاء في الأساسات الملامسة للتربة هو 50 مم."),
    ("متى تفك الدعامات السفلية للبلاطات؟", "تفك الدعامات بعد 7 أيام إذا كان الجو حاراً."),
]
f = retrievers[best]
print("Numeric guard (answers are illustrative inputs written for the example):")
for q, ans in cases:
    src = DOCS[int(np.argmax(f(q)))]
    a, s = numbers(ans), numbers(src)
    bad = [n for n in a if n[0] not in {x[0] for x in s}]
    print("  Q: %s | answer numbers=%s | source numbers=%s -> %s" %
          (q, sorted(x[0] for x in a), sorted(x[0] for x in s), "FLAG: unsupported " + ",".join(n[0] for n in bad) if bad else "OK"))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
names = list(res); x = np.arange(len(names))
ax.bar(x - 0.18, [100 * res[n][0] for n in names], 0.36, label="hit@1", color="tab:blue")
ax.bar(x + 0.18, [100 * res[n][1] for n in names], 0.36, label="hit@3", color="tab:orange")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, fontsize=8); ax.set_ylabel("Questions answered (%)")
ax.set_ylim(0, 100); ax.legend(fontsize=8)
for i, n in enumerate(names):
    ax.text(i + 0.18, 100 * res[n][1] + 1.5, "%.0f" % (100 * res[n][1]), ha="center", fontsize=7)
plt.tight_layout(); plt.savefig("fig/fig17_1_retrieval.png", dpi=150); plt.close()

# Figure 17.2: RAG pipeline schematic
fig, ax = plt.subplots(figsize=(9, 2.6)); ax.axis("off")
boxes = [("Engineer's\nquestion", 0.06), ("Normalise +\nretrieve top-3", 0.25), ("Snippets with\nclause IDs", 0.44),
         ("LLM drafts answer\nciting snippets", 0.63), ("Numeric & citation\nguard", 0.80), ("Engineer\nreviews & signs", 0.95)]
for t, xc in boxes:
    ax.add_patch(plt.Rectangle((xc - 0.075, 0.3), 0.15, 0.4, fill=True, fc="#e8f0fb", ec="tab:blue"))
    ax.text(xc, 0.5, t, ha="center", va="center", fontsize=8)
for (_, a), (_, b) in zip(boxes[:-1], boxes[1:]):
    ax.annotate("", xy=(b - 0.077, 0.5), xytext=(a + 0.077, 0.5), arrowprops=dict(arrowstyle="->"))
ax.annotate("", xy=(0.63, 0.28), xytext=(0.80, 0.28), arrowprops=dict(arrowstyle="->", color="tab:red", connectionstyle="arc3,rad=0.5"))
ax.text(0.715, 0.05, "flag -> regenerate or escalate", ha="center", fontsize=7, color="tab:red")
ax.set_xlim(0, 1.03); ax.set_ylim(0, 1)
plt.tight_layout(); plt.savefig("fig/fig17_2_pipeline.png", dpi=150); plt.close()

# site-jargon glossary (query expansion): colloquial/borrowed term -> formal term in the documents
GLOSS = {"سلامب": "هبوط", "كفر": "الغطاء الخرساني", "باطون": "الخرسانة", "طوبار": "قوالب", "شميت": "مطرقة الارتداد",
         "هامر": "مطرقة", "فايبريتر": "الهزاز", "مي": "ماء", "بالمي": "بالماء", "شوالات": "أكياس الإسمنت", "القوائم": "الدعامات",
         "التغطية": "الغطاء الخرساني", "نرش": "معالجة", "قوام": "هبوط", "الصبة": "الخرسانة", "البئر": "مياه الخلط",
         "الشمس": "الطقس الحار", "شاحنة": "الخلط", "نكب": "السقوط الحر"}
def expand(q): return q + " " + " ".join(v for k, v in GLOSS.items() if k in q)
f = retrievers[best]
h1 = h3 = 0
for q, gold in QA:
    o = np.argsort(-f(expand(q))); h1 += o[0] == gold; h3 += gold in o[:3]
print("  %-28s hit@1=%.2f  hit@3=%.2f  (glossary of %d terms, written after seeing the questions -> optimistic)" %
      ("Char+norm+site glossary", h1 / len(QA), h3 / len(QA), len(GLOSS)))